# Part 7 — Applicability Domain & Virtual Screening

**Goal:** Define the chemical space where the model's predictions are
trustworthy, then use it to screen candidate compounds.

## What is the Applicability Domain (AD)?

A QSAR model is not equally reliable across all possible molecules.
The AD is the region of chemical space where the model has enough
structural information to make confident predictions.
Predictions *outside* the AD should be treated as extrapolations — the
model is predicting molecular behaviour for a structural class it has
never seen.

This is especially important for EGFR because the three inhibitor
generations are structurally distinct:
- **1st gen** (Erlotinib, Gefitinib) — quinazoline scaffolds, reversible
- **2nd gen** (Afatinib) — acrylamide warhead, covalent
- **3rd gen** (Osimertinib) — pyrimidine-indole, targets T790M mutation

A model trained mostly on 1st gen compounds will have a narrow AD that
excludes 3rd gen structures. The Williams plot and UMAP will show this.

## Two AD methods used here
1. **Leverage / Williams plot** — classical QSAR AD based on hat matrix diagonal
2. **Tanimoto distance to training set centroid** — fingerprint-based AD,
   more interpretable chemically

**Inputs:** `best_model_part5.pkl`, `X_ecfp6.csv`, `y_pchembl.csv`,
`egfr_bioactivity_cleaned.csv`, `butina_cluster_ids_reg.npy`

**Outputs:** 4 figures, `part7_virtual_screen_hits.csv`

---
## 0. Installs & Imports

In [ ]:
# Uncomment on first Colab run:
# !pip install rdkit-pypi umap-learn chembl_webresource_client

import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
from sklearn.model_selection import BaseCrossValidator
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import umap

print('All imports OK')

---
## 1. Load Data

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
with open('best_model_part5.pkl', 'rb') as f:
    model = pickle.load(f)
print(f'Model loaded: {type(model).__name__}')

# ── Feature matrix ────────────────────────────────────────────────────────────
X_df    = pd.read_csv('X_ecfp6.csv')
mol_ids = X_df['molecule_chembl_id'].reset_index(drop=True)
X       = X_df.drop(columns='molecule_chembl_id').values.astype(np.float32)

# ── Targets ───────────────────────────────────────────────────────────────────
y_pchembl_df = pd.read_csv('y_pchembl.csv').set_index('molecule_chembl_id')
y_reg        = y_pchembl_df.loc[mol_ids, 'pchembl_value'].values

# ── Cleaned dataset (for SMILES and class labels) ─────────────────────────────
df_clean = pd.read_csv('egfr_bioactivity_cleaned.csv').set_index('molecule_chembl_id')
df_clean = df_clean.loc[mol_ids].reset_index()

# ── Butina cluster IDs ────────────────────────────────────────────────────────
cluster_ids = np.load('butina_cluster_ids_reg.npy')

print(f'X shape       : {X.shape}')
print(f'Compounds     : {len(mol_ids)}')
print(f'Class balance : {df_clean["bioactivity_class"].value_counts().to_dict()}')

In [ ]:
# ── Build Mol objects once (used in multiple cells below) ─────────────────────
mols = [Chem.MolFromSmiles(smi) for smi in df_clean['canonical_smiles']]
n_invalid = sum(1 for m in mols if m is None)
print(f'Valid molecules: {len(mols) - n_invalid} / {len(mols)}')

# ── Compute ECFP6 as RDKit bit vectors (needed for Tanimoto) ─────────────────
fpg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
fps = [fpg.GetFingerprint(mol) for mol in mols if mol is not None]

---
## 2. Split — Last Butina Fold as Test Set

For the AD analysis we need a genuine held-out test set.
We use the last fold from our Butina CV as a proxy — it is the
structurally most distant set from the training data by construction.

In [ ]:
class ButinaCrossValidator(BaseCrossValidator):
    def __init__(self, n_splits=5):
        self.n_splits = n_splits
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits
    def _iter_test_masks(self, X=None, y=None, groups=None):
        unique_cids = np.unique(groups)
        sizes       = {c: np.sum(groups == c) for c in unique_cids}
        sorted_cids = sorted(unique_cids, key=lambda c: -sizes[c])
        fold_assign = {c: i % self.n_splits for i, c in enumerate(sorted_cids)}
        for fold in range(self.n_splits):
            yield np.array([fold_assign[c] == fold for c in groups], dtype=bool)

cv = ButinaCrossValidator(n_splits=5)
splits = list(cv.split(X, y_reg, groups=cluster_ids))
train_idx, test_idx = splits[-1]   # last fold = most novel test set

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_reg[train_idx], y_reg[test_idx]

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Test fold: {len(test_idx)} compounds')
print(f'R² = {r2:.3f}   RMSE = {rmse:.3f} pChEMBL')

---
## 3. Applicability Domain — Method 1: Leverage (Williams Plot)

**Theory:**
The leverage of a compound measures how "unusual" it is relative
to the training set in feature space.

For a feature matrix X, the hat matrix is:
```
H = X (X'X)⁻¹ X'
```
The diagonal `h_i` is the leverage of compound i.
High leverage = far from the training data centroid in ECFP6 space.

**Williams plot:**
Scatter of standardised residuals (y-axis) vs leverage (x-axis).
The warning threshold is:
```
h* = 3(k+1)/n   where k = features, n = training set size
```
Compounds with h > h* are **outside the AD** — predictions are extrapolations.

**Note on dimensionality:**
With 2048 ECFP6 bits, the full hat matrix is computationally heavy.
We use the first 50 principal components as a proxy — this captures
the dominant variance while keeping matrix inversion tractable.

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 50 PCs for tractable leverage computation
N_PCS = 50
pca   = PCA(n_components=N_PCS, random_state=42)
Z_train = pca.fit_transform(X_train)
Z_test  = pca.transform(X_test)

# Hat matrix diagonal for training set
# h_i = z_i^T (Z'Z)^{-1} z_i
ZtZ     = Z_train.T @ Z_train
ZtZ_inv = np.linalg.pinv(ZtZ)    # pseudoinverse is safer than inv for near-singular matrices

def leverage(Z_row, ZtZ_inv):
    return float(Z_row @ ZtZ_inv @ Z_row)

h_train = np.array([leverage(Z_train[i], ZtZ_inv) for i in range(len(Z_train))])
h_test  = np.array([leverage(Z_test[i],  ZtZ_inv) for i in range(len(Z_test))])

# Warning threshold
k     = N_PCS
n     = len(Z_train)
h_star = 3 * (k + 1) / n

# Standardised residuals for test set
residuals     = y_test - y_pred
std_residuals = (residuals - residuals.mean()) / residuals.std()

n_outside_ad  = (h_test > h_star).sum()
pct_outside   = 100 * n_outside_ad / len(h_test)

print(f'AD threshold h* = {h_star:.4f}')
print(f'Test compounds outside AD: {n_outside_ad} / {len(h_test)} ({pct_outside:.1f}%)')

In [ ]:
# ── Figure 1: Williams Plot ────────────────────────────────────────────────────
inside_ad  = h_test <= h_star
outside_ad = ~inside_ad

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(h_test[inside_ad],  std_residuals[inside_ad],
           alpha=0.5, s=20, color='steelblue', label=f'Inside AD (n={inside_ad.sum()})')
ax.scatter(h_test[outside_ad], std_residuals[outside_ad],
           alpha=0.7, s=30, color='red', marker='x',
           label=f'Outside AD (n={outside_ad.sum()}, {pct_outside:.1f}%)')

ax.axvline(h_star, color='red', linestyle='--', linewidth=1.2,
           label=f'h* = {h_star:.3f}')
ax.axhline( 3, color='orange', linestyle=':', linewidth=1,  label='±3σ residual boundary')
ax.axhline(-3, color='orange', linestyle=':', linewidth=1)
ax.axhline( 0, color='black',  linestyle='-', linewidth=0.5)

ax.set_xlabel('Leverage (h)', fontsize=11)
ax.set_ylabel('Standardised Residual', fontsize=11)
ax.set_title('Williams Plot — EGFR pChEMBL Prediction\n'
             'Red crosses: outside applicability domain',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('part7_williams_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved part7_williams_plot.png')

In [ ]:
# ── R² inside vs outside AD ───────────────────────────────────────────────────
if inside_ad.sum() > 5:
    r2_in  = r2_score(y_test[inside_ad],  y_pred[inside_ad])
    r2_out = r2_score(y_test[outside_ad], y_pred[outside_ad]) if outside_ad.sum() > 5 else np.nan
    print(f'R² inside  AD ({inside_ad.sum()} compounds) : {r2_in:.3f}')
    print(f'R² outside AD ({outside_ad.sum()} compounds): {r2_out:.3f}')
    print()
    print('Expected: R² inside AD > R² outside AD')
    print('If not: the leverage-based AD may not be the right metric for sparse binary fingerprints.')

---
## 4. Applicability Domain — Method 2: Tanimoto Distance

A more chemically intuitive AD: a test compound is inside the AD
if its maximum Tanimoto similarity to any training compound exceeds
a threshold (typically 0.4–0.6).

This directly encodes the medicinal chemistry intuition:
"the model is reliable for compounds similar to what it was trained on."

In [ ]:
# Recompute RDKit bit-vector fingerprints (needed for BulkTanimotoSimilarity)
fps_train = [fpg.GetFingerprint(mols[i]) for i in train_idx if mols[i] is not None]
fps_test  = [fpg.GetFingerprint(mols[i]) for i in test_idx  if mols[i] is not None]

# For each test compound: maximum Tanimoto similarity to any training compound
TANIMOTO_THRESHOLD = 0.4   # compounds with max_sim < 0.4 are outside the Tanimoto AD

max_sim = np.array([
    max(DataStructs.BulkTanimotoSimilarity(fp_te, fps_train))
    for fp_te in fps_test
])

inside_tanimoto  = max_sim >= TANIMOTO_THRESHOLD
outside_tanimoto = ~inside_tanimoto

pct_in = 100 * inside_tanimoto.sum() / len(inside_tanimoto)
print(f'Tanimoto AD (threshold = {TANIMOTO_THRESHOLD})')
print(f'  Inside  AD: {inside_tanimoto.sum()} / {len(inside_tanimoto)} ({pct_in:.1f}%)')
print(f'  Outside AD: {outside_tanimoto.sum()} / {len(inside_tanimoto)} ({100-pct_in:.1f}%)')

if inside_tanimoto.sum() > 5 and outside_tanimoto.sum() > 5:
    r2_tan_in  = r2_score(y_test[:len(fps_test)][inside_tanimoto],
                           y_pred[:len(fps_test)][inside_tanimoto])
    r2_tan_out = r2_score(y_test[:len(fps_test)][outside_tanimoto],
                           y_pred[:len(fps_test)][outside_tanimoto])
    print(f'  R² inside  Tanimoto AD: {r2_tan_in:.3f}')
    print(f'  R² outside Tanimoto AD: {r2_tan_out:.3f}')

---
## 5. Figure 2 — UMAP Chemical Space

UMAP reduces the 2048-bit ECFP6 space to 2D, preserving local
structure (similar molecules cluster together).

**What to look for:**
- Dense blue clusters = large scaffold families (quinazolines, pyrimidines)
- Isolated orange dots (inactive) at the periphery = structurally atypical
- FDA drugs (stars) should be in the densest active region
- Test fold compounds (triangles) scattered away from the training mass
  confirms that Butina CV creates a genuine structural split

In [ ]:
print('Computing UMAP embedding... (may take 1-3 min)')
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='jaccard',    # Tanimoto for binary fingerprints
    random_state=42
)
embedding = reducer.fit_transform(X)
print(f'UMAP embedding shape: {embedding.shape}')

In [ ]:
# ── Figure 2: UMAP coloured by pChEMBL value ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Panel A: colour by pChEMBL ────────────────────────────────────────────────
sc = axes[0].scatter(
    embedding[:, 0], embedding[:, 1],
    c=y_reg, cmap='RdYlGn', alpha=0.5, s=12, vmin=4, vmax=10
)
plt.colorbar(sc, ax=axes[0], label='pChEMBL (pIC50)')

# Mark train/test split
axes[0].scatter(embedding[test_idx, 0], embedding[test_idx, 1],
                s=40, facecolors='none', edgecolors='blue',
                linewidths=0.8, alpha=0.7, label='Test fold')

# Mark FDA-approved drugs
FDA_DRUGS = {
    'Erlotinib'  : 'C#Cc1cccc(Nc2ncnc3cc(OCCO)c(OCCO)cc23)c1',
    'Gefitinib'  : 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
    'Osimertinib': 'C=CC(=O)Nc1cc2c(Nc3ccc(N(C)CCN(C)C)c(OC)c3)ncnc2cn1C',
}

for drug_name, smiles in FDA_DRUGS.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp_drug = fpg.GetFingerprint(mol)
        fp_array = np.array(fp_drug, dtype=np.float32).reshape(1, -1)
        drug_umap = reducer.transform(fp_array)[0]
        axes[0].scatter(*drug_umap, s=120, marker='*',
                        color='red', zorder=5, edgecolors='darkred')
        axes[0].annotate(drug_name, drug_umap, fontsize=7,
                         xytext=(5, 3), textcoords='offset points',
                         color='darkred', fontweight='bold')

axes[0].set_title('UMAP — ECFP6 Chemical Space\nColoured by pChEMBL value',
                  fontweight='bold')
axes[0].legend(fontsize=8, markerscale=1.5)
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

# ── Panel B: colour by bioactivity class ──────────────────────────────────────
class_colours = {'active': '#2196F3', 'inactive': '#FF9800', 'intermediate': '#9E9E9E'}
for cls, colour in class_colours.items():
    mask = df_clean['bioactivity_class'] == cls
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1],
                    c=colour, alpha=0.4, s=12, label=f'{cls} (n={mask.sum()})')

axes[1].set_title('UMAP — ECFP6 Chemical Space\nColoured by bioactivity class',
                  fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig('part7_umap_chemical_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved part7_umap_chemical_space.png')

---
## 6. Figure 3 — AD Coverage by Bioactivity Class

Are certain activity classes more likely to be outside the AD?
If all out-of-AD compounds are active, it may indicate that the
most potent scaffolds are underrepresented in the training data.

In [ ]:
# Combine AD flags with class labels for the test set
test_classes = df_clean.iloc[test_idx]['bioactivity_class'].values

ad_df = pd.DataFrame({
    'bioactivity_class': test_classes,
    'inside_leverage_AD': inside_ad,
    'max_tanimoto': max_sim[:len(test_idx)],
    'inside_tanimoto_AD': inside_tanimoto[:len(test_idx)],
    'y_true': y_test,
    'y_pred': y_pred,
    'residual': y_test - y_pred
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: % inside AD by class (leverage)
ad_by_class = ad_df.groupby('bioactivity_class')['inside_leverage_AD'].mean() * 100
ad_by_class.plot(kind='bar', ax=axes[0], color=['#2196F3', '#9E9E9E', '#FF9800'],
                  edgecolor='white', rot=0)
axes[0].set_ylabel('% compounds inside AD')
axes[0].set_title('Leverage AD Coverage by Class', fontweight='bold')
axes[0].set_ylim(0, 110)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9)

# Panel B: Max Tanimoto similarity distribution by class
for cls, colour in [('active', '#2196F3'), ('inactive', '#FF9800')]:
    mask = ad_df['bioactivity_class'] == cls
    if mask.sum() > 0:
        axes[1].hist(ad_df.loc[mask, 'max_tanimoto'], bins=20, alpha=0.6,
                     color=colour, label=cls, edgecolor='white')
axes[1].axvline(TANIMOTO_THRESHOLD, color='red', linestyle='--',
                label=f'AD threshold ({TANIMOTO_THRESHOLD})')
axes[1].set_xlabel('Max Tanimoto similarity to training set')
axes[1].set_ylabel('Compounds')
axes[1].set_title('Tanimoto Similarity Distribution by Class', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('part7_ad_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved part7_ad_coverage.png')

---
## 7. Mini Virtual Screen

We query ChEMBL for EGFR compounds measured with **Ki** (binding affinity)
instead of IC50. These are structurally related but not in our training set
(which used only IC50 with `assay_type='B'`).

For each candidate:
1. Predict pChEMBL
2. Check if inside the Tanimoto AD
3. Rank by predicted pChEMBL (descending)

Only report predictions inside the AD — anything outside is flagged
as an extrapolation.

In [ ]:
from chembl_webresource_client.new_client import new_client

print('Querying ChEMBL for EGFR Ki compounds (not in training set)...')

activity_client = new_client.activity
# Ki measurements give independent validation — different assay type
res = activity_client.filter(
    target_chembl_id='CHEMBL203',
    standard_type='Ki',
    assay_type='B'
)
df_ki = pd.DataFrame.from_dict(res)

# Apply the same quality filters as Part 1
df_ki = df_ki[
    df_ki['pchembl_value'].notnull() &
    (df_ki['potential_duplicate'] == 0) &
    (df_ki['standard_relation'].isin(['=', "'="]))
].copy()
df_ki['pchembl_value'] = pd.to_numeric(df_ki['pchembl_value'])

# Remove any that were already in our training set
df_ki = df_ki[~df_ki['molecule_chembl_id'].isin(mol_ids)].copy()

# Validate SMILES
df_ki['mol'] = df_ki['canonical_smiles'].apply(Chem.MolFromSmiles)
df_ki = df_ki[df_ki['mol'].notna()].copy()

print(f'Ki compounds found (not in training set): {len(df_ki)}')

In [ ]:
if len(df_ki) == 0:
    print('No Ki compounds found — using a small set of known inhibitors instead.')
    # Fallback: use known EGFR inhibitors from literature with confirmed activity
    fallback_smiles = [
        ('Lapatinib',    'CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1'),
        ('Neratinib',    'C=CC(=O)Nc1cccc(Nc2ncnc3cc(Oc4ccc(N5CCN(C)CC5)cc4)c(Cl)cc23)c1'),
        ('Dacomitinib',  'C=CC(=O)N1CCC[C@@H]1c1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC'),
        ('Poziotinib',   'C=CC(=O)Nc1ccc2ncnc(Nc3ccc(OC)c(Cl)c3)c2c1'),
        ('Mobocertinib', 'CCC(=O)Nc1cc2c(Nc3cc(C)c(NC(=O)C=C)cc3OC)ncnc2cc1OC'),
        ('Canertinib',   'Cl.Cl.FC(F)(F)c1cc2c(Nc3ccc4c(cnn4Cc4ccccn4)c3)ncnc2cc1Br'),
    ]
    df_ki = pd.DataFrame(fallback_smiles, columns=['molecule_pref_name', 'canonical_smiles'])
    df_ki['pchembl_value'] = np.nan   # unknown — we're predicting these
    df_ki['mol'] = df_ki['canonical_smiles'].apply(Chem.MolFromSmiles)
    df_ki = df_ki[df_ki['mol'].notna()].copy()
    print(f'Using {len(df_ki)} fallback EGFR inhibitors')

In [ ]:
# ── Featurize screening candidates ────────────────────────────────────────────
screen_fps  = [fpg.GetFingerprint(mol) for mol in df_ki['mol']]
screen_X    = np.array([list(fp) for fp in screen_fps], dtype=np.float32)

# ── Predict pChEMBL ───────────────────────────────────────────────────────────
screen_pred = model.predict(screen_X)

# ── AD check: max Tanimoto to training set ────────────────────────────────────
# Train on the FULL dataset for screening (not just fold 4)
model.fit(X, y_reg)
fps_all_train = [fpg.GetFingerprint(mol) for mol in mols if mol is not None]

screen_max_sim = np.array([
    max(DataStructs.BulkTanimotoSimilarity(fp, fps_all_train))
    for fp in screen_fps
])

# ── Build results table ───────────────────────────────────────────────────────
name_col = 'molecule_pref_name' if 'molecule_pref_name' in df_ki.columns else 'molecule_chembl_id'

screen_results = pd.DataFrame({
    'compound'         : df_ki[name_col].fillna(df_ki.get('molecule_chembl_id', 'unknown')).values,
    'predicted_pchembl': screen_pred.round(3),
    'experimental_pchembl': df_ki['pchembl_value'].values,
    'max_tanimoto_to_train': screen_max_sim.round(3),
    'inside_AD'        : (screen_max_sim >= TANIMOTO_THRESHOLD),
    'predicted_class'  : np.where(screen_pred >= 6.0, 'active',
                         np.where(screen_pred <= 5.0, 'inactive', 'intermediate')),
    'smiles'           : df_ki['canonical_smiles'].values,
}).sort_values('predicted_pchembl', ascending=False).reset_index(drop=True)

print('=== VIRTUAL SCREENING RESULTS ===')
print(screen_results[['compound', 'predicted_pchembl', 'experimental_pchembl',
                       'max_tanimoto_to_train', 'inside_AD',
                       'predicted_class']].to_string(index=False))

screen_results.to_csv('part7_virtual_screen_hits.csv', index=False)
print('\nSaved part7_virtual_screen_hits.csv')

---
## 8. Figure 4 — Virtual Screen Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Predicted pChEMBL bar chart, coloured by AD status
colors_ad = ['#2196F3' if inside else '#FF5722'
             for inside in screen_results['inside_AD']]
bars = axes[0].barh(
    screen_results['compound'],
    screen_results['predicted_pchembl'],
    color=colors_ad, edgecolor='white', height=0.6
)
axes[0].axvline(6.0, color='green', linestyle='--', linewidth=1.2,
                label='Active threshold (pChEMBL = 6)')
axes[0].axvline(5.0, color='orange', linestyle=':', linewidth=1,
                label='Inactive threshold (pChEMBL = 5)')

# Add actual values if available
exp_vals = screen_results['experimental_pchembl']
if exp_vals.notna().any():
    axes[0].scatter(exp_vals, range(len(screen_results)),
                    color='black', marker='|', s=80, zorder=5,
                    linewidths=2, label='Experimental pChEMBL')

# Custom legend for AD colours
from matplotlib.patches import Patch
ad_legend = [Patch(color='#2196F3', label='Inside AD'),
             Patch(color='#FF5722', label='Outside AD (extrapolation)')]
axes[0].legend(handles=ad_legend + axes[0].get_legend_handles_labels()[0], fontsize=7)
axes[0].set_xlabel('Predicted pChEMBL')
axes[0].set_title('Virtual Screen — Predicted Potency\n(Blue = inside AD, Red = extrapolation)',
                  fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Panel B: Tanimoto similarity to training set
axes[1].barh(
    screen_results['compound'],
    screen_results['max_tanimoto_to_train'],
    color=colors_ad, edgecolor='white', height=0.6
)
axes[1].axvline(TANIMOTO_THRESHOLD, color='red', linestyle='--', linewidth=1.2,
                label=f'AD threshold ({TANIMOTO_THRESHOLD})')
axes[1].set_xlabel('Max Tanimoto similarity to training set')
axes[1].set_title('Structural Similarity to Training Data\n(Determines AD coverage)',
                  fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, 1.05)

plt.tight_layout()
plt.savefig('part7_virtual_screen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved part7_virtual_screen.png')

---
## Summary

### Files produced
| File | What it shows |
|---|---|
| `part7_williams_plot.png` | Classical QSAR AD: leverage vs residuals |
| `part7_umap_chemical_space.png` | Full dataset in 2D fingerprint space |
| `part7_ad_coverage.png` | AD coverage by bioactivity class |
| `part7_virtual_screen.png` | Screening predictions with AD flags |
| `part7_virtual_screen_hits.csv` | Full screening table |

### Key numbers for the README (fill in yours)
```
Applicability domain: XX% of test compounds inside Tanimoto AD (threshold = 0.4).
R² inside AD = X.XXX vs R² outside AD = X.XXX — confirming that predictions
are more reliable for structurally familiar compounds.
Virtual screen of N EGFR Ki compounds identified M predicted actives
(pChEMBL ≥ 6.0) within the AD.
```

### Limitations to mention in the README
- AD is estimated, not computed exactly — both methods are proxies
- Tanimoto threshold (0.4) is a heuristic; the optimal value is dataset-dependent
- The model was trained on IC50 data; predicting from Ki data involves
  an assay-type assumption that is not validated here
- No experimental validation of the virtual screen hits

### Next: Part 8 — Streamlit App & Docker
Packages the model + AD check into a web app:
input SMILES → predicted pChEMBL + activity class + AD flag.